# Uncertainty And Sensitivity Analysis

This notebook is the final workflow step. It starts from the spatial MRV products created in notebook `07` and explores how sensitive estimated CO2 removal is to uncertain assumptions.

The goal is not to replace full SCEPTER uncertainty runs. It gives us a transparent first-pass uncertainty envelope around spatial MRV estimates, so we can see which assumptions matter most and where better field or lab data would reduce uncertainty.


## Workflow

1. **Load spatial MRV results** from `data/outputs/maps/spatial_mrv/spatial_mrv_scepter_results.gpkg`.
2. **Choose a base CO2 metric** if one is available. If not, create a clearly marked placeholder from area and application-rate assumptions so the notebook structure can be tested.
3. **Define uncertain parameters**: application-rate multiplier, particle-size multiplier, runoff multiplier, soil-pH effect, and basalt-chemistry factor.
4. **Run Monte Carlo scenarios** for every model-unit/scenario result row.
5. **Summarize uncertainty** with p05, p50, p95, mean, and coefficient of variation.
6. **Rank sensitivity drivers** using correlation between sampled parameters and modeled CO2 removal.
7. **Export tables, maps, figures, and a Markdown uncertainty report**.

Once full SCEPTER ensembles are available, this notebook should be updated to read the ensemble outputs directly rather than scaling a base metric.


In [ ]:
from pathlib import Path
import os
import sys

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def mount_google_drive_if_colab() -> None:
    try:
        from google.colab import drive
    except ModuleNotFoundError:
        return

    drive.mount("/content/drive")


mount_google_drive_if_colab()

COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/erw_spatial_mrv")
COLAB_DATA_ROOT = COLAB_PROJECT_ROOT / "data"
LOCAL_PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()


def has_erw_package(project_root: Path) -> bool:
    return (project_root / "src" / "erw_mrv" / "__init__.py").exists()


def source_root_candidates() -> list[Path]:
    cwd = Path.cwd().resolve()
    candidates = [LOCAL_PROJECT_ROOT, COLAB_PROJECT_ROOT]
    for base in (cwd, *cwd.parents):
        candidates.extend((base, base / "erw_spatial_mrv"))
    candidates.extend(
        Path(path)
        for path in (
            "/content/erw_spatial_mrv",
            "/content/enhanced_rock_weathering/erw_spatial_mrv",
            "/content/drive/MyDrive/erw_spatial_mrv",
        )
    )
    unique = []
    for candidate in candidates:
        if candidate not in unique:
            unique.append(candidate)
    return unique


def find_source_project_root() -> Path:
    for candidate in source_root_candidates():
        if has_erw_package(candidate):
            return candidate
    checked = chr(10).join(f"- {candidate}" for candidate in source_root_candidates())
    raise ModuleNotFoundError(
        "Could not find src/erw_mrv. The data can live in Google Drive, but "
        "the notebook still needs the project source folder containing src/erw_mrv. "
        "In Colab, upload/sync the full erw_spatial_mrv project or run from a "
        "checkout that includes src/. Checked: "
        f"{checked}"
    )


SOURCE_PROJECT_ROOT = find_source_project_root()
PROJECT_ROOT = COLAB_PROJECT_ROOT if COLAB_DATA_ROOT.exists() else SOURCE_PROJECT_ROOT
DATA_ROOT = COLAB_DATA_ROOT if COLAB_DATA_ROOT.exists() else PROJECT_ROOT / "data"
os.environ["ERW_MRV_DATA_ROOT"] = str(DATA_ROOT)

SRC = SOURCE_PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"SOURCE_PROJECT_ROOT = {SOURCE_PROJECT_ROOT}")
print(f"DATA_ROOT = {DATA_ROOT}")

from erw_mrv.paths import OUTPUT_FIGURES, OUTPUT_MAPS, OUTPUT_REPORTS, OUTPUT_TABLES, ensure_dir

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 160)
plt.rcParams["figure.figsize"] = (10, 8)


## Configure Inputs And Outputs


In [ ]:
SPATIAL_MRV_DIR = OUTPUT_MAPS / "spatial_mrv"
SPATIAL_RESULTS_PATH = SPATIAL_MRV_DIR / "spatial_mrv_scepter_results.gpkg"

UNCERTAINTY_MAP_DIR = ensure_dir(OUTPUT_MAPS / "uncertainty")
UNCERTAINTY_FIGURE_DIR = ensure_dir(OUTPUT_FIGURES / "uncertainty")
UNCERTAINTY_TABLE_DIR = ensure_dir(OUTPUT_TABLES / "uncertainty")
UNCERTAINTY_REPORT_DIR = ensure_dir(OUTPUT_REPORTS / "uncertainty")

if not SPATIAL_RESULTS_PATH.exists():
    raise FileNotFoundError(
        "Missing spatial MRV result layer. Run notebook 07 first. "
        f"Expected: {SPATIAL_RESULTS_PATH}"
    )

SPATIAL_RESULTS_PATH


## Load Spatial MRV Results


In [ ]:
spatial_results = gpd.read_file(SPATIAL_RESULTS_PATH)
print(f"Spatial result rows: {len(spatial_results):,}")
print(f"Scenarios: {spatial_results['scenario_id'].nunique() if 'scenario_id' in spatial_results.columns else 0:,}")
spatial_results.head()


## Select Or Build Base CO2 Metric

If parsed SCEPTER outputs already include a CO2 removal metric, we use it. If not, this creates a placeholder estimate from area and basalt application rate so the uncertainty machinery can be tested. Placeholder values are clearly marked and should not be reported as measured/modelled removal.


In [ ]:
CO2_METRIC_CANDIDATES = [
    "additional_co2_t",
    "co2_removal_t",
    "cumulative_co2_removal_t",
    "net_co2_removal_t",
    "co2_removed_t",
]
base_metric = next(
    (
        column for column in CO2_METRIC_CANDIDATES
        if column in spatial_results.columns and pd.api.types.is_numeric_dtype(spatial_results[column])
    ),
    None,
)

results = spatial_results.copy()
if base_metric is None:
    BASE_REMOVAL_FRACTION_OF_APPLIED_BASALT = 0.05
    if "basalt_application_t_ha" not in results.columns:
        results["basalt_application_t_ha"] = 0.0
    results["base_co2_removal_t"] = (
        results["area_ha"].fillna(0)
        * results["basalt_application_t_ha"].fillna(0)
        * BASE_REMOVAL_FRACTION_OF_APPLIED_BASALT
    )
    results["base_metric_source"] = "placeholder_area_rate_fraction"
    base_metric = "base_co2_removal_t"
else:
    results["base_co2_removal_t"] = results[base_metric].fillna(0)
    results["base_metric_source"] = f"parsed_{base_metric}"

print(f"Base metric: {base_metric}")
results[["run_id", "scenario_id", "area_ha", "basalt_application_t_ha", "base_co2_removal_t", "base_metric_source"]].head()


## Define Uncertain Parameters

These ranges are first-pass assumptions. Tighten them with measured basalt chemistry, local runoff/rainfall, lab weathering data, and SCEPTER ensemble runs.


In [ ]:
RANDOM_SEED = 42
N_DRAWS = 500
rng = np.random.default_rng(RANDOM_SEED)

PARAMETERS = {
    "application_rate_multiplier": {"low": 0.85, "mode": 1.00, "high": 1.15},
    "particle_size_multiplier": {"low": 0.75, "mode": 1.00, "high": 1.35},
    "runoff_multiplier": {"low": 0.70, "mode": 1.00, "high": 1.30},
    "soil_ph_effect": {"low": 0.85, "mode": 1.00, "high": 1.10},
    "basalt_chemistry_factor": {"low": 0.80, "mode": 1.00, "high": 1.20},
}

parameter_table = pd.DataFrame(
    [{"parameter": name, **values} for name, values in PARAMETERS.items()]
)
parameter_table_path = UNCERTAINTY_TABLE_DIR / "uncertainty_parameter_ranges.csv"
parameter_table.to_csv(parameter_table_path, index=False)
parameter_table


## Monte Carlo Sampling

The scaling equation is deliberately transparent:

`uncertain_co2 = base_co2 * application_rate * runoff * soil_pH * basalt_chemistry / particle_size`

Smaller particle size generally increases weathering potential, so the particle-size multiplier is placed in the denominator.


In [ ]:
def triangular_sample(low, mode, high, size, rng):
    return rng.triangular(low, mode, high, size=size)

samples = pd.DataFrame(
    {
        name: triangular_sample(values["low"], values["mode"], values["high"], N_DRAWS, rng)
        for name, values in PARAMETERS.items()
    }
)
samples["combined_multiplier"] = (
    samples["application_rate_multiplier"]
    * samples["runoff_multiplier"]
    * samples["soil_ph_effect"]
    * samples["basalt_chemistry_factor"]
    / samples["particle_size_multiplier"]
)

samples.head()


## Apply Uncertainty To Spatial Results


In [ ]:
records = []
for _, row in results.drop(columns="geometry").iterrows():
    base = float(row.get("base_co2_removal_t", 0) or 0)
    draws = base * samples["combined_multiplier"].to_numpy()
    records.append(
        {
            "run_id": row.get("run_id"),
            "model_unit_id": row.get("model_unit_id"),
            "scenario_id": row.get("scenario_id"),
            "area_ha": row.get("area_ha"),
            "base_co2_removal_t": base,
            "co2_p05_t": float(np.percentile(draws, 5)),
            "co2_p50_t": float(np.percentile(draws, 50)),
            "co2_p95_t": float(np.percentile(draws, 95)),
            "co2_mean_t": float(np.mean(draws)),
            "co2_std_t": float(np.std(draws)),
            "co2_cv": float(np.std(draws) / np.mean(draws)) if np.mean(draws) else np.nan,
            "base_metric_source": row.get("base_metric_source"),
        }
    )

uncertainty = pd.DataFrame(records)
uncertainty_path = UNCERTAINTY_TABLE_DIR / "scepter_uncertainty_by_run.csv"
uncertainty.to_csv(uncertainty_path, index=False)
uncertainty.head()


## Scenario-Level Uncertainty Summary


In [ ]:
scenario_uncertainty = uncertainty.groupby("scenario_id").agg(
    run_count=("run_id", "count"),
    area_ha=("area_ha", "sum"),
    base_co2_t=("base_co2_removal_t", "sum"),
    co2_p05_t=("co2_p05_t", "sum"),
    co2_p50_t=("co2_p50_t", "sum"),
    co2_p95_t=("co2_p95_t", "sum"),
    mean_cv=("co2_cv", "mean"),
).reset_index()

scenario_uncertainty_path = UNCERTAINTY_TABLE_DIR / "scepter_uncertainty_by_scenario.csv"
scenario_uncertainty.to_csv(scenario_uncertainty_path, index=False)
scenario_uncertainty


## Sensitivity Ranking

This estimates which uncertain inputs drive the spread in CO2 removal most strongly. It uses rank correlation against the combined multiplier, which is a useful first check before full model-based sensitivity analysis.


In [ ]:
sensitivity_rows = []
for parameter in PARAMETERS:
    sensitivity_rows.append(
        {
            "parameter": parameter,
            "spearman_like_rank_correlation": samples[parameter].rank().corr(samples["combined_multiplier"].rank()),
            "pearson_correlation": samples[parameter].corr(samples["combined_multiplier"]),
        }
    )

sensitivity = pd.DataFrame(sensitivity_rows).sort_values(
    "spearman_like_rank_correlation",
    key=lambda s: s.abs(),
    ascending=False,
)
sensitivity_path = UNCERTAINTY_TABLE_DIR / "scepter_uncertainty_sensitivity_ranking.csv"
sensitivity.to_csv(sensitivity_path, index=False)
sensitivity


## Export Spatial Uncertainty Layer


In [ ]:
spatial_uncertainty = results[["run_id", "geometry"]].merge(uncertainty, on="run_id", how="left")
spatial_uncertainty_path = UNCERTAINTY_MAP_DIR / "scepter_uncertainty_by_run.gpkg"
spatial_uncertainty.to_file(spatial_uncertainty_path, layer="uncertainty_by_run", driver="GPKG")
spatial_uncertainty_path


## Uncertainty Maps

Maps use median CO2 removal and uncertainty width (`p95 - p05`). If outputs are placeholders, treat the maps as workflow QA only.


In [ ]:
map_paths = []
for column, label in [("co2_p50_t", "median_co2"), ("co2_p95_t", "p95_co2")]:
    if column not in spatial_uncertainty.columns:
        continue
    fig, ax = plt.subplots(figsize=(10, 8))
    spatial_uncertainty.plot(column=column, ax=ax, legend=True, cmap="viridis", linewidth=0.1, edgecolor="black")
    ax.set_title(f"Uncertainty map: {column}")
    ax.set_axis_off()
    out_path = UNCERTAINTY_MAP_DIR / f"uncertainty_{label}.png"
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    map_paths.append(out_path)

spatial_uncertainty["co2_uncertainty_width_t"] = spatial_uncertainty["co2_p95_t"] - spatial_uncertainty["co2_p05_t"]
fig, ax = plt.subplots(figsize=(10, 8))
spatial_uncertainty.plot(column="co2_uncertainty_width_t", ax=ax, legend=True, cmap="magma", linewidth=0.1, edgecolor="black")
ax.set_title("Uncertainty width: p95 - p05")
ax.set_axis_off()
width_map_path = UNCERTAINTY_MAP_DIR / "uncertainty_width_p95_minus_p05.png"
fig.savefig(width_map_path, dpi=200, bbox_inches="tight")
plt.show()
map_paths.append(width_map_path)
map_paths


## Scenario Uncertainty Figure


In [ ]:
plot_data = scenario_uncertainty.sort_values("co2_p50_t", ascending=False).copy()
x = np.arange(len(plot_data))
y = plot_data["co2_p50_t"].to_numpy()
yerr = np.vstack([
    y - plot_data["co2_p05_t"].to_numpy(),
    plot_data["co2_p95_t"].to_numpy() - y,
])

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x, y, yerr=yerr, capsize=4)
ax.set_xticks(x)
ax.set_xticklabels(plot_data["scenario_id"], rotation=35, ha="right")
ax.set_ylabel("CO2 removal estimate (t)")
ax.set_title("Scenario uncertainty range")
fig.tight_layout()
scenario_figure_path = UNCERTAINTY_FIGURE_DIR / "scenario_uncertainty_ranges.png"
fig.savefig(scenario_figure_path, dpi=200, bbox_inches="tight")
plt.show()
scenario_figure_path


## Sensitivity Figure


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
plot_sensitivity = sensitivity.copy()
ax.barh(plot_sensitivity["parameter"], plot_sensitivity["spearman_like_rank_correlation"])
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Rank correlation with CO2 multiplier")
ax.set_title("First-pass sensitivity ranking")
fig.tight_layout()
sensitivity_figure_path = UNCERTAINTY_FIGURE_DIR / "sensitivity_ranking.png"
fig.savefig(sensitivity_figure_path, dpi=200, bbox_inches="tight")
plt.show()
sensitivity_figure_path


## Write Uncertainty Report


In [ ]:
report_path = UNCERTAINTY_REPORT_DIR / "uncertainty_sensitivity_report.md"
map_list = "\n".join(f"- `{path.relative_to(PROJECT_ROOT)}`" for path in map_paths)
placeholder_note = "placeholder" if results["base_metric_source"].astype(str).str.contains("placeholder").any() else "parsed SCEPTER metric"

report = f"""# Uncertainty And Sensitivity Report

## Inputs

- Spatial MRV layer: `{SPATIAL_RESULTS_PATH.relative_to(PROJECT_ROOT)}`
- Base metric: `{base_metric}`
- Base metric source: `{placeholder_note}`
- Monte Carlo draws: {N_DRAWS:,}

## Main Outputs

- Run-level uncertainty table: `{uncertainty_path.relative_to(PROJECT_ROOT)}`
- Scenario uncertainty table: `{scenario_uncertainty_path.relative_to(PROJECT_ROOT)}`
- Sensitivity ranking table: `{sensitivity_path.relative_to(PROJECT_ROOT)}`
- Spatial uncertainty layer: `{spatial_uncertainty_path.relative_to(PROJECT_ROOT)}`
- Scenario uncertainty figure: `{scenario_figure_path.relative_to(PROJECT_ROOT)}`
- Sensitivity figure: `{sensitivity_figure_path.relative_to(PROJECT_ROOT)}`

## Maps

{map_list}

## Interpretation Notes

If the base metric source is `placeholder_area_rate_fraction`, these outputs are workflow QA only. They should be replaced by uncertainty analysis based on parsed SCEPTER ensemble outputs once the model execution pipeline is connected.

The first-pass sensitivity ranking indicates which assumptions have the strongest effect in the current scaling model. Prioritize measured data for the highest-ranked drivers.
"""
report_path.write_text(report, encoding="utf-8")
print(report_path)
print(report[:2000])


## Outputs From This Notebook

This notebook writes uncertainty and sensitivity products under `data/outputs/`:

- `data/outputs/tables/uncertainty/uncertainty_parameter_ranges.csv`
- `data/outputs/tables/uncertainty/scepter_uncertainty_by_run.csv`
- `data/outputs/tables/uncertainty/scepter_uncertainty_by_scenario.csv`
- `data/outputs/tables/uncertainty/scepter_uncertainty_sensitivity_ranking.csv`
- `data/outputs/maps/uncertainty/scepter_uncertainty_by_run.gpkg`
- `data/outputs/maps/uncertainty/uncertainty_median_co2.png`
- `data/outputs/maps/uncertainty/uncertainty_p95_co2.png`
- `data/outputs/maps/uncertainty/uncertainty_width_p95_minus_p05.png`
- `data/outputs/figures/uncertainty/scenario_uncertainty_ranges.png`
- `data/outputs/figures/uncertainty/sensitivity_ranking.png`
- `data/outputs/reports/uncertainty/uncertainty_sensitivity_report.md`

This closes the first end-to-end workflow. The next improvement is to replace placeholder/scaled uncertainty with true SCEPTER ensemble outputs and measured soil, climate, basalt chemistry, and field validation data.
